In [1452]:
import os
import re
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

load_dotenv()

True

In [1453]:
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_APIKEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

In [1454]:
db_fetched_data = """
오늘 체온 36.7도. 대변은 안 보고 방귀만 엄청 뽕뽕 뀜. 오늘 완모 수유는 양쪽 합쳐서 15분씩 총 7번 직수함. 먹다가 자꾸 젖을 빼고 짜증 내서 배앓이인가 싶어 트림 열심히 시켜줌. 낮잠은 유모차 태워서 동네 한 바퀴 도니까 그나마 40분 잠듦. 오후에 뒤집기 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘림. 밤 9시에 막수하고 잠들었는데 제발 새벽에 한 번만 깨자.
"""

In [1455]:
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

step1_insights = ""
for i, line in enumerate(lines):
    step1_insights += f"{i+1}. {line}\n"
step1_insights = step1_insights.strip()


In [1456]:
extract_params = {
    GenParams.DECODING_METHOD: "greedy",
    GenParams.TEMPERATURE: 0.0,
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 1500  
}

extractor_model = ModelInference(
    model_id="ibm/granite-4-h-small", 
    credentials=credentials,
    params=extract_params,
    project_id=project_id
)


In [1457]:
extract_prompt = f"""[Instruction] 
당신은 텍스트 분석 및 정보 추출 전문가입니다. 
주어진 [Data]의 텍스트를 줄바꿈(\\n)과 문장부호(., !, ?) 기준으로 빠짐없이 분리하세요.
그 후, 원문 문장 순서를 그대로 유지하면서 아래의 [Output Format]에 맞춰 각 문장을 분석한 결과를 출력하세요.

각 문장 분석 시 아래 규칙을 철저히 따르세요:
1. 핵심명사: 문장의 주제가 되는 주요 고유명사나 일반명사 (없으면 '없음')
2. 행동명사: 문장에서 일어나는 행위, 동작, 상태 변화를 나타내는 명사 (예: 구매, 이동, 회의, 취소 등 / 없으면 '없음')
3. 수치+단위: 문장에 포함된 숫자와 단위의 조합 (예: 3명, 11시, 50,000원 / 없으면 '없음')
4. 예측 감정단어: 문맥의 흐름과 뉘앙스를 파악하여 작성자의 심리 상태나 감정을 나타내는 단어 1개 (예: 기대, 불안, 만족, 다급, 평온 등)

[Data]
{step1_insights}

[Output Format]
1. [첫 번째 문장 원문]
- 핵심명사: 
- 행동명사: 
- 수치+단위: 
- 예측 감정단어: 

2. [두 번째 문장 원문]
- 핵심명사: 
- 행동명사: 
- 수치+단위: 
- 예측 감정단어: 

[Answer]:"""


In [1458]:
extract_response = extractor_model.generate(prompt=extract_prompt)
results_list = extract_response.get('results', [])
first_result = next(iter(results_list)) if isinstance(results_list, list) else results_list
step2_keywords = first_result.get('generated_text', '').strip()

In [1459]:
creative_params = {
    GenParams.DECODING_METHOD: "sample",
    GenParams.TEMPERATURE: 0.7,
    GenParams.TOP_P: 0.85,
    GenParams.MIN_NEW_TOKENS: 50,
    GenParams.MAX_NEW_TOKENS: 600  
}

writer_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct", 
    credentials=credentials,
    params=creative_params,
    project_id=project_id
)

In [1460]:
diary_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
너는 인스타그램에 다정하고 따뜻한 육아 에세이를 연재하는 대한민국의 '엄마'이다.
입력된 데이터의 숫자와 팩트를 바탕으로, 아래의 [강제 출력 규칙]을 결합하여 완벽한 문장 흐름의 에세이를 출력해라.

[강제 출력 규칙 - 100% 절대 준수]
1. [텍스트로만 시작]: 문장의 첫 시작은 무조건 순수한 한글 단어로만 시작해라. 대괄호`[]`, 기호, 숫자, `1번 일기` 같은 템플릿 문구는 출력물 전체에서 완전히 제외해라.
2. [오직 엄마(나) 주어 고정]: 모든 문장은 엄마가 행동하고, 엄마가 관찰하고, 엄마가 느끼는 시점으로만 작성해라. (예: '내가 온도를 재보니', '내가 수유를 해주니', '내가 관찰하니 느껴진 속마음')
3. [줄바꿈 및 문장 수 일치]: 마침표(.)가 끝날 때마다 무조건 엔터(줄바꿈)를 입력해라. 전체 줄 수는 입력된 메모의 문장 개수와 정확히 동일하게 맞추어라.
4. [숫자 기호 그대로 유지]: 본문의 숫자와 단위 기호(36.7도, 15분, 7번, 40분, 9시, 한 번 등)는 가공하지 말고 형태 그대로 100% 노출해라.
5. [다정한 구어체 어미 교차]: 모든 문장의 끝은 반드시 "~했어요.", "~했답니다.", "~하네요.", "~봅니다." 중 하나로 끝맺어라. 각 문장마다 종결 어미를 다채롭게 교차하여 리듬감을 살려라.
6. [문장 전면 재창조]: 원문의 기계적인 조사나 어색한 단어 나열을 완전히 해체해라. 2단계의 숨은 감정 흐름(평온, 불안, 만족, 다급, 안도, 노력, 기대)을 엄마의 부드러운 독백 어휘로 변환하여 문맥이 물 흐르듯 이어지게 윤문해라.

[지정 금지 단어]
'컨디션', '안정적', '모습', '순간', '울림', '가슴', '최고'는 본문에서 제외해라.

[1단계 요약 메모]
{step1_insights}

[Diary]:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""


In [1461]:
writer_response = writer_model.generate(prompt=diary_prompt)
writer_results = writer_response.get('results', [])
first_writer_result = next(iter(writer_results)) if isinstance(writer_results, list) else writer_results
raw_diary = first_writer_result.get('generated_text', '').strip()

In [1462]:
if "[END]" in raw_diary:
    raw_diary = raw_diary.split("[END]")[0].strip()

cleaned_diary = re.sub(r'[\u4e00-\u9fff]', '', raw_diary) 
cleaned_diary = re.sub(r'[^가-힣a-zA-Z0-9\s\.,!\?]', '', cleaned_diary) # 💡 'ml' 기호 보존용 필터 유지
cleaned_diary = re.sub(r'^\d+[\.\s\-~)]+', '', cleaned_diary, flags=re.MULTILINE)

cleaned_diary = re.sub(r'\.\s+', '.\n', cleaned_diary)
diary_lines = [line.strip() for line in cleaned_diary.split('\n') if line.strip()]

full_print_lines = []
for line in diary_lines:
    line = re.sub(r'^\d+[\.\s\-~)]+', '', line).strip()
    if line:
        full_print_lines.append(line)

final_lines = []
for line in full_print_lines:
    if len(line.encode('utf-8')) > 230:
        while len(line.encode('utf-8')) > 227:
            line = line[:-1]
        line = line.strip() + "..."
    final_lines.append(line)

final_diary = "\n".join(final_lines[:])

In [1463]:
print("\n=== 1단계: 구조화된 요약 메모 추출 완료 ===")
print(step1_insights)


=== 1단계: 구조화된 요약 메모 추출 완료 ===
1. 오늘 체온 36.7도.
2. 대변은 안 보고 방귀만 엄청 뽕뽕 뀜.
3. 오늘 완모 수유는 양쪽 합쳐서 15분씩 총 7번 직수함.
4. 먹다가 자꾸 젖을 빼고 짜증 내서 배앓이인가 싶어 트림 열심히 시켜줌.
5. 낮잠은 유모차 태워서 동네 한 바퀴 도니까 그나마 40분 잠듦.
6. 오후에 뒤집기 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘림.
7. 밤 9시에 막수하고 잠들었는데 제발 새벽에 한 번만 깨자.


In [1464]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 오늘 체온 36.7도.
- 핵심명사: 체온
- 행동명사: 없음
- 수치+단위: 36.7도
- 예측 감정단어: 평온

2. 대변은 안 보고 방귀만 엄청 뽕뽕 뀜.
- 핵심명사: 대변, 방귀
- 행동명사: 뽕뽕 뀜
- 수치+단위: 없음
- 예측 감정단어: 불안

3. 오늘 완모 수유는 양쪽 합쳐서 15분씩 총 7번 직수함.
- 핵심명사: 완모 수유
- 행동명사: 직수함
- 수치+단위: 15분, 7번
- 예측 감정단어: 만족

4. 먹다가 자꾸 젖을 빼고 짜증 내서 배앓이인가 싶어 트림 열심히 시켜줌.
- 핵심명사: 젖, 배앓이, 트림
- 행동명사: 빼고, 내서, 시켜줌
- 수치+단위: 없음
- 예측 감정단어: 다급

5. 낮잠은 유모차 태워서 동네 한 바퀴 도니까 그나마 40분 잠듦.
- 핵심명사: 낮잠, 유모차
- 행동명사: 태워서, 도니까, 잠듦
- 수치+단위: 40분
- 예측 감정단어: 평온

6. 오후에 뒤집기 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘림.
- 핵심명사: 뒤집기, 허리
- 행동명사: 들썩들썩하느라, 흘림
- 수치+단위: 없음
- 예측 감정단어: 불안

7. 밤 9시에 막수하고 잠들었는데 제발 새벽에 한 번만 깨자.
- 핵심명사: 막수, 잠들었, 새벽
- 행동명사: 없음
- 수치+단위: 9시
- 예측 감정단어: 기대


In [1465]:
print("\n=== 3단계: 최종 완성된 감성 일기 ===")

for idx, final_line in enumerate(full_print_lines):
    print(f"[{idx+1}번 일기]: {final_line}")
print(f"\n-> 최종 결과물 총 문장 수: {len(full_print_lines)}줄")


=== 3단계: 최종 완성된 감성 일기 ===
[1번 일기]: 오늘은 아이의 체온을 재보니 36.7도였어요.
[2번 일기]: 아이가 대변은 보지 않고 방귀만 엄청 뽕뽕 뀜 하는 모습을 보니까 속이 편안했답니다.
[3번 일기]: 오늘은 완모 수유를 양쪽 합쳐서 15분씩 총 7번 직수해줬어요.
[4번 일기]: 아이가 먹다가 자꾸 젖을 빼고 짜증 내서 배앓이인가 싶어 트림을 열심히 시켜줬더니 조금 편해진 것 같아요.
[5번 일기]: 낮잠은 유모차에 태워서 동네 한 바퀴 돌렸더니 그나마 40분 잠을 잤어요.
[6번 일기]: 오후에 뒤집기 하려고 허리를 들썩들썩하느라 땀을 한 바가지 흘렸네요.
[7번 일기]: 밤 9시에 막수하고 잠들었는데 제발 새벽에 한 번만 깨면 좋겠어요.

-> 최종 결과물 총 문장 수: 7줄
